In [8]:
# ==============================================================================
# NB_CARATULAS_CALCULATION_PLANIFICADOR
# SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP CON PLANIFICADORES
# ==============================================================================
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, coalesce, greatest, round, lead, lag, row_number, desc,
    max as f_max, min as f_min, explode, array
)
from pyspark.sql.window import Window

print("=== SNIPPET 1: CARGA Y BOOTSTRAP CON PLANIFICADORES ===")

fecha_actual = datetime.now()
ano_actual = int(fecha_actual.strftime("%Y"))
print(f"Año de ejecución: {ano_actual}")

# Cargar tabla de planificadores
df_master_basico_raw = spark.sql("""
    SELECT * 
    FROM LH_CARATULAS_PLNF.continental_master_planificadores
""")

print(f"Registros cargados (raw): {df_master_basico_raw.count():,}")

# Deduplicación por clave primaria CON PLANIFICADOR
w_dedup = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano", "ORDENM", "PWOH").orderBy("Mes")
df_master_basico = (
    df_master_basico_raw
    .withColumn("rn", row_number().over(w_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f"Registros tras deduplicación: {df_master_basico.count():,}")

# Sample inicial por planificador
print("\nSAMPLE INICIAL - CR-LAVA por planificador:")
(df_master_basico
 .filter((col("PAIS") == "CR") & (col("Familia") == "LAVA") & (col("Ano") == 2025))
 .groupBy("Planificador")
 .agg(
     F.sum("Inv_Inicial").alias("Total_Inv_Inicial"),
     F.sum("Llegadas").alias("Total_Llegadas"),
     F.sum("Fcst").alias("Total_Fcst")
 )
 .orderBy("Planificador").show(10, truncate=False))

print("[COMPLETADO] Bootstrap con planificadores")

StatementMeta(, a90d9477-282f-494f-9090-bb14a582417b, 10, Finished, Available, Finished)

=== SNIPPET 1: CARGA Y BOOTSTRAP CON PLANIFICADORES ===
Año de ejecución: 2025
Registros cargados (raw): 26,856
Registros tras deduplicación: 26,856

SAMPLE INICIAL - CR-LAVA por planificador:
+------------+-----------------+--------------+----------+
|Planificador|Total_Inv_Inicial|Total_Llegadas|Total_Fcst|
+------------+-----------------+--------------+----------+
|D5T         |0.0              |0.0           |0.0       |
|D5U         |264.0            |20.0          |96.0      |
|D5V         |0.0              |0.0           |0.0       |
|D5Y         |49408.0          |3110.0        |23472.0   |
|D5Z         |1241.0           |0.0           |75.0      |
|F5F         |0.0              |0.0           |0.0       |
|F5Q         |1.0              |0.0           |0.0       |
|G4I         |0.0              |0.0           |0.0       |
+------------+-----------------+--------------+----------+

[COMPLETADO] Bootstrap con planificadores


In [9]:
# ==============================================================================
# SNIPPET 2: DETECCIÓN DEL MES ANCLA POR PLANIFICADOR
# ==============================================================================
print("=== SNIPPET 2: DETECCIÓN MES ANCLA POR PLANIFICADOR ===")

print("LÓGICA: Mes ancla = ÚLTIMO mes con Facturación > 0 POR PLANIFICADOR")

# Ventana por año Y PLANIFICADOR
w_año_planif = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano")

df_con_ancla = (
    df_master_basico
    .withColumn("mes_candidato_ancla", 
                when(coalesce(col("Facturacion"), lit(0.0)) > 0.0, col("ORDENM")))
    .withColumn("mes_ancla", f_max("mes_candidato_ancla").over(w_año_planif))
    .withColumn("mes_ancla", coalesce(col("mes_ancla"), lit(1)))
    .drop("mes_candidato_ancla")
    .withColumn("es_mes_ancla", col("ORDENM") == col("mes_ancla"))
    .withColumn("antes_del_ancla", col("ORDENM") < col("mes_ancla"))
    .withColumn("despues_del_ancla", col("ORDENM") > col("mes_ancla"))
)

print("\nVerificación anclas por planificador:")
(df_con_ancla.filter(col("es_mes_ancla") == True)
 .select("PAIS", "Familia", "Planificador", "Ano", "mes_ancla", "Attribute", 
         F.round("Facturacion", 0).alias("Facturacion"))
 .distinct().orderBy("PAIS", "Familia", "Planificador", "Ano").show(15, truncate=False))

print("[COMPLETADO] Detección ancla por planificador")

StatementMeta(, a90d9477-282f-494f-9090-bb14a582417b, 11, Finished, Available, Finished)

=== SNIPPET 2: DETECCIÓN MES ANCLA POR PLANIFICADOR ===
LÓGICA: Mes ancla = ÚLTIMO mes con Facturación > 0 POR PLANIFICADOR

Verificación anclas por planificador:
+----+-------+------------+----+---------+---------+-----------+
|PAIS|Familia|Planificador|Ano |mes_ancla|Attribute|Facturacion|
+----+-------+------------+----+---------+---------+-----------+
|CL  |AIAC   |D5Q         |2025|1        |ENE      |0.0        |
|CL  |AIAC   |D5Q         |2026|1        |ENE      |0.0        |
|CL  |AIAC   |D5Q         |2027|1        |ENE      |0.0        |
|CL  |AIAC   |D5Q         |2028|1        |ENE      |0.0        |
|CL  |AIAC   |D5Q         |2029|1        |ENE      |0.0        |
|CL  |AIAC   |D5Q         |2030|1        |ENE      |0.0        |
|CL  |AIAC   |F5J         |2025|1        |ENE      |0.0        |
|CL  |AIAC   |F5J         |2026|1        |ENE      |0.0        |
|CL  |AIAC   |F5J         |2027|1        |ENE      |0.0        |
|CL  |AIAC   |F5J         |2028|1        |ENE      |0.0  

In [10]:
# ==============================================================================
# VERSIÓN PLANIFICADORES CLEAN (FACTURADO)
# SNIPPET 3: LLEGADAS E INVENTARIOS POR PLANIFICADOR - VERSIÓN CLEAN
# ==============================================================================
print("=== SNIPPET 3: LLEGADAS E INVENTARIOS POR PLANIFICADOR - VERSIÓN CLEAN ===")

print("LÓGICAS:")
print("1. ANTES del ancla: Calcular llegadas, mantener Inv_Final del Daily")
print("2. MES ANCLA: Inv_Final = Inv_Inicial + Llegadas - FACTURACIÓN")
print("3. DESPUÉS del ancla: Inv_Final = Inv_Inicial + Llegadas - S&OP")
print("4. Todo POR PLANIFICADOR")

# Ventana multi-año POR PLANIFICADOR
w_continuo_planif = Window.partitionBy("PAIS", "Familia", "Planificador").orderBy("Ano", "ORDENM")
w_año_planif = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano")

# Detectar último mes con S&OP POR PLANIFICADOR
df_con_limites = (
    df_con_ancla
    .withColumn("mes_candidato_fcst", 
                when(coalesce(col("Fcst"), lit(0.0)) > 0.0, col("ORDENM")))
    .withColumn("ultimo_mes_fcst", f_max("mes_candidato_fcst").over(w_año_planif))
    .withColumn("ultimo_mes_fcst", coalesce(col("ultimo_mes_fcst"), lit(12)))
    .drop("mes_candidato_fcst")
)

df_calculado = df_con_limites

for iteracion in range(1, 6):
    print(f"  Iteración {iteracion} por planificador...")
    
    # Continuidad POR PLANIFICADOR
    df_calculado = (
        df_calculado
        .withColumn("Inv_Final_anterior", lag("Inv_Final").over(w_continuo_planif))
        .withColumn(
            "Inv_Inicial_nuevo",
            when(col("ORDENM") == 1,
                 coalesce(col("Inv_Final_anterior"), col("Inv_Inicial")))
            .otherwise(coalesce(col("Inv_Final_anterior"), col("Inv_Inicial")))
        )
        .drop("Inv_Final_anterior")
    )
    
    # Calcular llegadas ANTES del ancla
    df_calculado = df_calculado.withColumn(
        "Llegadas_nuevo",
        when(
            col("antes_del_ancla") & (coalesce(col("Llegadas"), lit(0.0)) == 0.0),
            greatest(
                lit(0.0),
                coalesce(col("Inv_Final"), lit(0.0)) -
                coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) +
                coalesce(col("Facturacion"), lit(0.0))
            )
        ).otherwise(coalesce(col("Llegadas"), lit(0.0)))
    )
    
    # Recalcular Inv_Final - VERSIÓN CLEAN (CON FACTURACIÓN EN MES ANCLA)
    df_calculado = df_calculado.withColumn(
        "Inv_Final_nuevo",
        when(
            # MES ANCLA: USAR FACTURACIÓN
            col("es_mes_ancla"),
            greatest(
                lit(0.0),
                coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) +
                coalesce(col("Llegadas_nuevo"), lit(0.0)) -
                coalesce(col("Facturacion"), lit(0.0))
            )
        ).when(
            # DESPUÉS DEL ANCLA: USAR S&OP
            col("despues_del_ancla") & 
            (col("ORDENM") <= col("ultimo_mes_fcst")) &
            (coalesce(col("Fcst"), lit(0.0)) > 0.0),
            greatest(
                lit(0.0),
                coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) +
                coalesce(col("Llegadas_nuevo"), lit(0.0)) -
                coalesce(col("Fcst"), lit(0.0))
            )
        ).otherwise(
            # ANTES del ancla: Mantener del Daily
            coalesce(col("Inv_Final"), lit(0.0))
        )
    )
    
    df_calculado = (
        df_calculado
        .drop("Inv_Inicial", "Llegadas", "Inv_Final")
        .withColumnRenamed("Inv_Inicial_nuevo", "Inv_Inicial")
        .withColumnRenamed("Llegadas_nuevo", "Llegadas")
        .withColumnRenamed("Inv_Final_nuevo", "Inv_Final")
    )

df_master_corregido = (
    df_calculado
    .select(
        col("Ano"), col("Mes"), col("PAIS"), col("Familia"), col("Planificador"),
        col("Attribute"), col("ORDENM"), col("Inv_Inicial"), col("Llegadas"), 
        col("Fcst"), col("Facturacion"), col("Inv_Final"),
        col("mes_ancla"), col("ultimo_mes_fcst"), col("es_mes_ancla"), 
        col("antes_del_ancla"), col("despues_del_ancla")
    )
).cache()

print(f"Tabla corregida por planificador (CLEAN): {df_master_corregido.count():,}")
print("[COMPLETADO] Cálculos por planificador - VERSIÓN CLEAN")


StatementMeta(, a90d9477-282f-494f-9090-bb14a582417b, 12, Finished, Available, Finished)

=== SNIPPET 3: LLEGADAS E INVENTARIOS POR PLANIFICADOR - VERSIÓN CLEAN ===
LÓGICAS:
1. ANTES del ancla: Calcular llegadas, mantener Inv_Final del Daily
2. MES ANCLA: Inv_Final = Inv_Inicial + Llegadas - FACTURACIÓN
3. DESPUÉS del ancla: Inv_Final = Inv_Inicial + Llegadas - S&OP
4. Todo POR PLANIFICADOR
  Iteración 1 por planificador...
  Iteración 2 por planificador...
  Iteración 3 por planificador...
  Iteración 4 por planificador...
  Iteración 5 por planificador...
Tabla corregida por planificador (CLEAN): 26,856
[COMPLETADO] Cálculos por planificador - VERSIÓN CLEAN


In [11]:
# ==============================================================================
# SNIPPET 4: LOOP PWOH 1-10 POR PLANIFICADOR
# ==============================================================================
print("=== SNIPPET 4: LOOP PWOH POR PLANIFICADOR ===")

# Cross join PWOH
pwoh_array = [lit(i) for i in range(1, 11)]
df_pwoh_base = (
    df_master_corregido
    .withColumn("pwoh_array", array(*pwoh_array))
    .withColumn("PWOH", explode(col("pwoh_array")))
    .drop("pwoh_array")
)

# Ventanas POR PLANIFICADOR
w_intra_planif = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano", "PWOH").orderBy("ORDENM")
w_multi_planif = Window.partitionBy("PAIS", "Familia", "Planificador", "PWOH").orderBy("Ano", "ORDENM")

df_pwoh_final = (
    df_pwoh_base
    .withColumn("fcst_sig_intra", lead("Fcst", 1).over(w_intra_planif))
    .withColumn("fcst_sig_multi", lead("Fcst", 1).over(w_multi_planif))
    .withColumn("aplica_pwoh", 
                col("despues_del_ancla") & 
                (col("ORDENM") <= col("ultimo_mes_fcst")) &
                (coalesce(col("Llegadas"), lit(0.0)) == 0.0))
    .withColumn(
        "Llegadas_pwoh",
        when(
            col("aplica_pwoh"),
            when(
                col("ORDENM") == 12,
                greatest(lit(0.0),
                    ((coalesce(col("Fcst"), lit(0.0)) + 
                      coalesce(col("fcst_sig_multi"), col("Fcst"), lit(0.0))) / 4.0 * col("PWOH")) +
                    coalesce(col("Fcst"), lit(0.0)) - coalesce(col("Inv_Inicial"), lit(0.0))
                )
            ).otherwise(
                greatest(lit(0.0),
                    (coalesce(col("fcst_sig_intra"), lit(0.0)) / 4.0 * col("PWOH")) +
                    coalesce(col("Fcst"), lit(0.0)) - coalesce(col("Inv_Inicial"), lit(0.0))
                )
            )
        ).otherwise(coalesce(col("Llegadas"), lit(0.0)))
    )
    .withColumn(
        "Inv_Final_pwoh",
        when(
            col("aplica_pwoh"),
            greatest(lit(0.0),
                coalesce(col("Inv_Inicial"), lit(0.0)) +
                coalesce(col("Llegadas_pwoh"), lit(0.0)) -
                coalesce(col("Fcst"), lit(0.0))
            )
        ).otherwise(coalesce(col("Inv_Final"), lit(0.0)))
    )
    .select("Ano", "Mes", "PAIS", "Familia", "Planificador", "Attribute", "ORDENM", "PWOH",
            col("Inv_Inicial").alias("Inv_Inic"),
            col("Llegadas_pwoh").alias("Llegadas"),
            "Fcst", "Facturacion",
            col("Inv_Final_pwoh").alias("Inv_Final"),
            "mes_ancla", "aplica_pwoh")
).cache()

print(f"PWOH por planificador: {df_pwoh_final.count():,}")
print("[COMPLETADO] Loop PWOH por planificador")

StatementMeta(, a90d9477-282f-494f-9090-bb14a582417b, 13, Finished, Available, Finished)

=== SNIPPET 4: LOOP PWOH POR PLANIFICADOR ===
PWOH por planificador: 268,560
[COMPLETADO] Loop PWOH por planificador


In [12]:
# ==============================================================================
# SNIPPET 5: CÁLCULO TTOS POR PLANIFICADOR
# ==============================================================================
print("=== SNIPPET 5: TTOS POR PLANIFICADOR ===")

w_intra_ttos_planif = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano", "PWOH").orderBy("ORDENM")
w_multi_ttos_planif = Window.partitionBy("PAIS", "Familia", "Planificador", "PWOH").orderBy("Ano", "ORDENM")

df_ttos = (
    df_pwoh_final
    .withColumn("lleg_m1_intra", lead("Llegadas", 1).over(w_intra_ttos_planif))
    .withColumn("lleg_m2_intra", lead("Llegadas", 2).over(w_intra_ttos_planif))
    .withColumn("lleg_m1_multi", lead("Llegadas", 1).over(w_multi_ttos_planif))
    .withColumn("lleg_m2_multi", lead("Llegadas", 2).over(w_multi_ttos_planif))
    .withColumn(
        "TTOS",
        when(
            col("ORDENM") == 11,
            coalesce(col("lleg_m1_intra"), lit(0.0)) + 
            (coalesce(col("lleg_m2_multi"), col("lleg_m1_intra"), lit(0.0)) / 2.0)
        ).when(
            col("ORDENM") == 12,
            coalesce(col("lleg_m1_multi"), col("lleg_m1_intra"), lit(0.0)) +
            (coalesce(col("lleg_m2_multi"), col("lleg_m2_intra"), col("lleg_m1_multi"), lit(0.0)) / 2.0)
        ).otherwise(
            coalesce(col("lleg_m1_intra"), lit(0.0)) + 
            (coalesce(col("lleg_m2_intra"), lit(0.0)) / 2.0)
        )
    )
    .withColumn("UND_TTOS", col("Inv_Final") + col("TTOS"))
    .select("Ano", "Mes", "PAIS", "Familia", "Planificador", "Attribute", "ORDENM", "PWOH",
            "Inv_Inic", "Llegadas", "Fcst", "Facturacion", "Inv_Final", 
            "TTOS", "UND_TTOS")
).cache()

print(f"TTOS por planificador: {df_ttos.count():,}")
print("[COMPLETADO] TTOS por planificador")

StatementMeta(, a90d9477-282f-494f-9090-bb14a582417b, 14, Finished, Available, Finished)

=== SNIPPET 5: TTOS POR PLANIFICADOR ===
TTOS por planificador: 268,560
[COMPLETADO] TTOS por planificador


In [13]:
# ==============================================================================
# SNIPPET 6: WOH POR PLANIFICADOR
# ==============================================================================
print("=== SNIPPET 6: WOH POR PLANIFICADOR ===")

# Último mes facturado POR PLANIFICADOR
df_ultimo_fact = (
    df_ttos.filter(col("Facturacion") > 0)
    .groupBy("PAIS", "Familia", "Planificador", "Ano", "PWOH")
    .agg(f_max("ORDENM").alias("ultimo_fact"))
)

# Fallbacks POR PLANIFICADOR
df_fallbacks = (
    df_ttos.filter((col("Ano") == 2025) & col("ORDENM").isin(1, 2))
    .groupBy("PAIS", "Familia", "Planificador", "PWOH")
    .pivot("ORDENM", [1, 2]).agg(F.first("Fcst"))
    .withColumnRenamed("1", "fcst_ene_2025").withColumnRenamed("2", "fcst_feb_2025")
)

w_intra_woh_planif = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano", "PWOH").orderBy("ORDENM")

df_woh = (
    df_ttos
    .join(df_ultimo_fact, ["PAIS", "Familia", "Planificador", "Ano", "PWOH"], "left")
    .join(df_fallbacks, ["PAIS", "Familia", "Planificador", "PWOH"], "left")
    .withColumn("ultimo_fact", coalesce(col("ultimo_fact"), lit(0)))
    .fillna(0, ["fcst_ene_2025", "fcst_feb_2025"])
    .withColumn("fcst_m1", lead("Fcst", 1).over(w_intra_woh_planif))
    .withColumn("fcst_m2", lead("Fcst", 2).over(w_intra_woh_planif))
    .withColumn(
        "denominador",
        when(
            col("ORDENM") <= col("ultimo_fact"),
            when(col("ORDENM") == 11,
                (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_ene_2025"), lit(0.0))) / 8.0
            ).when(col("ORDENM") == 12,
                (coalesce(col("fcst_ene_2025"), lit(0.0)) + coalesce(col("fcst_feb_2025"), lit(0.0))) / 8.0
            ).otherwise(
                (coalesce(col("fcst_m1"), lit(0.0)) + coalesce(col("fcst_m2"), lit(0.0))) / 8.0
            )
        ).otherwise(
            when(col("ORDENM") == 12,
                coalesce(col("fcst_ene_2025"), lit(0.0)) / 4.0
            ).otherwise(
                coalesce(col("fcst_m1"), lit(0.0)) / 4.0
            )
        )
    )
    .withColumn(
        "WOH",
        when((col("Inv_Final") <= 0) | (col("denominador") <= 0), lit(0))
        .otherwise(F.round(col("Inv_Final") / col("denominador"), 0).cast("int"))
    )
    .withColumn(
        "WOH_TTOS", 
        when((col("UND_TTOS") <= 0) | (col("denominador") <= 0), lit(0))
        .otherwise(F.round(col("UND_TTOS") / col("denominador"), 0).cast("int"))
    )
    .select("Ano", "Mes", "PAIS", "Familia", "Planificador", "Attribute", "ORDENM", "PWOH",
            "Inv_Inic", "Llegadas", "Fcst", "Facturacion", "Inv_Final", 
            "TTOS", "UND_TTOS", "WOH", "WOH_TTOS")
).cache()

print(f"WOH por planificador: {df_woh.count():,}")
print("[COMPLETADO] WOH por planificador")

StatementMeta(, a90d9477-282f-494f-9090-bb14a582417b, 15, Finished, Available, Finished)

=== SNIPPET 6: WOH POR PLANIFICADOR ===
WOH por planificador: 268,560
[COMPLETADO] WOH por planificador


In [14]:
# ==============================================================================
# SNIPPET 7: MONETARIOS Y GUARDADO CON PLANIFICADORES - VERSIÓN CLEAN
# ==============================================================================
print("=== SNIPPET 7: MONETARIOS Y GUARDADO CON PLANIFICADORES - VERSIÓN CLEAN ===")

# Cargar BGT (SIN planificador - por País+Familia)
try:
    df_bgt_pp = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_PP.csv")
        .select(col("Pais").alias("PAIS"), col("Familia"), col("PP").cast("double").alias("BGT_PP"))
    )
    print("BGT_PP cargado")
except:
    df_bgt_pp = spark.createDataFrame([
        ("CR", "LAVA", 164.5), ("CO", "LAVA", 100.0)
    ], ["PAIS", "Familia", "BGT_PP"])

try:
    df_bgt_cont = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_CONT.csv")
        .select("PAIS", "Familia", "Attribute", col("ORDENM").cast("int"), 
                col("BGT_QTY").cast("double"), col("BGT_$").cast("double").alias("BGT_USD"))
    )
    print("BGT_CONT cargado")
except:
    df_bgt_cont = spark.createDataFrame([
        ("CR", "LAVA", "ENE", 1, 2550.0, 100000.0)
    ], ["PAIS", "Familia", "Attribute", "ORDENM", "BGT_QTY", "BGT_USD"])

# JOIN BGT (solo por País+Familia, NO por planificador)
df_con_monetarios = (
    df_woh
    .join(df_bgt_pp, ["PAIS", "Familia"], "left")
    .join(df_bgt_cont, ["PAIS", "Familia", "Attribute", "ORDENM"], "left")
    .fillna(0, ["BGT_PP", "BGT_QTY", "BGT_USD"])
    .withColumn("Inv_Final_USD", (col("Inv_Final") * col("BGT_PP")) / 1_000_000)
    .withColumn("TTOS_USD", (col("TTOS") * col("BGT_PP")) / 1_000_000)
    .withColumn("USD_OH_TTOS", col("Inv_Final_USD") + col("TTOS_USD"))
    .withColumn("Dif_Real_vs_BGT_QTY", col("UND_TTOS") - col("BGT_QTY"))
    .withColumn("Dif_Real_vs_BGT_USD", col("USD_OH_TTOS") - col("BGT_USD"))
)

print(f"Monetarios agregados: {df_con_monetarios.count():,} registros")

# Tabla final CON PLANIFICADOR Y COLUMNA CALCULO
df_tabla_final = (
    df_con_monetarios
    .select(
        col("Ano").cast("int"),
        col("Mes"),
        when(col("PAIS").isin("CO", "EC", "PE", "AR", "CL"), "Andino")
        .when(col("PAIS").isin("CR", "GT", "HN", "SV", "MX"), "CEAM")
        .when(col("PAIS").isin("CA"), "NorteAmerica")
        .otherwise("Otros").alias("Region"),
        col("PAIS"),
        col("Familia"),
        col("Planificador"),  # INCLUIR PLANIFICADOR
        col("Attribute"),
        col("Inv_Inic").cast("double"),
        col("Llegadas").cast("double"),
        col("Fcst").alias("S&PO").cast("double"),
        col("Facturacion").alias("Facturado").cast("double"),
        col("Inv_Final").cast("double"),
        F.round(col("Inv_Final_USD"), 4).alias("Inv_Final_$"),
        col("WOH").alias("WHO").cast("int"),
        col("TTOS").cast("double"),
        F.round(col("TTOS_USD"), 4).alias("TTOS_$"),
        F.round(col("USD_OH_TTOS"), 4).alias("USD_OH_TTOS"),
        col("UND_TTOS").alias("UND_TTO").cast("double"),
        col("WOH_TTOS").alias("WOH_TTO").cast("int"),
        col("BGT_QTY").cast("double"),
        col("BGT_USD").alias("BGT_$").cast("double"),
        col("Dif_Real_vs_BGT_QTY").cast("double"),
        col("Dif_Real_vs_BGT_USD").alias("Dif_Real_vs_BGT_$").cast("double"),
        col("PWOH").cast("double"),
        col("ORDENM").cast("double"),
        lit("Facturado").alias("Calculo")  # COLUMNA IDENTIFICADORA
    )
)

print(f"Tabla final estructurada: {df_tabla_final.count():,} registros")

# Deduplicar POR PLANIFICADOR
count_antes = df_tabla_final.count()

df_tabla_sin_duplicados = df_tabla_final.dropDuplicates([
    "PAIS", "Familia", "Planificador", "Ano", "ORDENM", "PWOH"
])

count_despues = df_tabla_sin_duplicados.count()
duplicados_eliminados = count_antes - count_despues

print(f"Registros antes: {count_antes:,}")
print(f"Registros después: {count_despues:,}")
print(f"Duplicados eliminados: {duplicados_eliminados:,}")

df_tabla_final_opt = df_tabla_sin_duplicados.repartition(200, "PAIS", "Familia", "Planificador", "PWOH")

# GUARDAR TABLA - VERSIÓN CLEAN
spark.sql("USE LH_CARATULAS_PLNF")
spark.sql("DROP TABLE IF EXISTS tbl_caratula_cdv_planificador_fact")

df_tabla_final_opt.write.format("delta").mode("overwrite").saveAsTable("tbl_caratula_cdv_planificador_fact")

print("\n✅ TABLA GUARDADA: tbl_caratula_cdv_planificador_fact")
print(f"   Registros finales: {count_despues:,}")
print("   Columna Calculo: 'Facturado'")

# Verificación
print("\nSample CR-LAVA por planificador (CLEAN):")
spark.sql("""
    SELECT Planificador, Attribute, ORDENM, Inv_Inic, Llegadas, `S&PO`, 
           Facturado, Inv_Final, WHO, TTOS, Calculo
    FROM LH_CARATULAS_PLNF.tbl_caratula_cdv_planificador_fact
    WHERE PAIS = 'CR' AND Familia = 'LAVA' AND Ano = 2025 AND PWOH = 1
    ORDER BY Planificador, ORDENM
    LIMIT 20
""").show(20, truncate=False)

print("\n" + "="*80)
print("PROCESO COMPLETADO - VERSIÓN PLANIFICADORES CLEAN (FACTURADO)")
print("="*80)
print("✓ Mes ancla: Usa FACTURACIÓN para Inv_Final")
print("✓ Después ancla: Usa S&OP para Inv_Final")
print("✓ Columna 'Calculo' = 'Facturado'")
print("✓ Tabla: tbl_caratula_cdv_planificador_fact")
print("="*80)

print("[COMPLETADO] Monetarios y guardado - VERSIÓN CLEAN CON PLANIFICADORES")

StatementMeta(, a90d9477-282f-494f-9090-bb14a582417b, 16, Finished, Available, Finished)

=== SNIPPET 7: MONETARIOS Y GUARDADO CON PLANIFICADORES - VERSIÓN CLEAN ===
BGT_PP cargado
BGT_CONT cargado
Monetarios agregados: 513,360 registros
Tabla final estructurada: 513,360 registros
Registros antes: 513,360
Registros después: 268,560
Duplicados eliminados: 244,800

✅ TABLA GUARDADA: tbl_caratula_cdv_planificador_fact
   Registros finales: 268,560
   Columna Calculo: 'Facturado'

Sample CR-LAVA por planificador (CLEAN):
+------------+---------+------+--------+--------+----+---------+---------+---+----+---------+
|Planificador|Attribute|ORDENM|Inv_Inic|Llegadas|S&PO|Facturado|Inv_Final|WHO|TTOS|Calculo  |
+------------+---------+------+--------+--------+----+---------+---------+---+----+---------+
|D5T         |ENE      |1.0   |0.0     |0.0     |0.0 |0.0      |0.0      |0  |0.0 |Facturado|
|D5T         |FEB      |2.0   |0.0     |0.0     |0.0 |0.0      |0.0      |0  |0.0 |Facturado|
|D5T         |MAR      |3.0   |0.0     |0.0     |0.0 |0.0      |0.0      |0  |0.0 |Facturado|
|D5

-------------

In [1]:
#==============================================================================
# SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP - PLANIFICADORES
# ==============================================================================
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, coalesce, greatest, round, lead, lag, row_number, desc,
    max as f_max, min as f_min, explode, array, count, countDistinct
)
from pyspark.sql.window import Window

print("="*80)
print("SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP - PLANIFICADORES")
print("="*80)

df_master_basico_raw = spark.sql("""
    SELECT * 
    FROM LH_CARATULAS_PLNF.continental_master_planificadores
""")

print(f"✓ Registros cargados (raw): {df_master_basico_raw.count():,}")

w_dedup = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano", "ORDENM", "PWOH").orderBy("Mes")

df_master_basico = (
    df_master_basico_raw
    .withColumn("rn", row_number().over(w_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)

count_antes = df_master_basico_raw.count()
count_despues = df_master_basico.count()
duplicados = count_antes - count_despues

print(f"✓ Registros antes: {count_antes:,}")
print(f"✓ Registros después: {count_despues:,}")
print(f"✓ Duplicados eliminados: {duplicados:,}")

df_master_basico = df_master_basico.cache()

print("\n[✓] SNIPPET 1 COMPLETADO")

StatementMeta(, 889760ac-d51b-408a-964a-45d2c97e687d, 3, Finished, Available, Finished)

SNIPPET 1: CARGA DE DATOS Y BOOTSTRAP - PLANIFICADORES
✓ Registros cargados (raw): 38,880
✓ Registros antes: 38,880
✓ Registros después: 38,880
✓ Duplicados eliminados: 0

[✓] SNIPPET 1 COMPLETADO


In [2]:
# ==============================================================================
# SNIPPET 2: DETECCIÓN DEL MES ANCLA GLOBAL
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 2: DETECCIÓN DEL MES ANCLA GLOBAL")
print("="*80)

w_global = Window.orderBy(lit(1))

df_con_ancla = (
    df_master_basico
    .withColumn("periodo_id", col("Ano") * 100 + col("ORDENM"))
    .withColumn("periodo_candidato", 
                when(coalesce(col("Facturacion"), lit(0.0)) > 0.0, col("periodo_id")))
    .withColumn("periodo_ancla", f_max("periodo_candidato").over(w_global))
    .withColumn("periodo_ancla", coalesce(col("periodo_ancla"), lit(202501)))
    .withColumn("ano_ancla", (col("periodo_ancla") / 100).cast("int"))
    .withColumn("mes_ancla", col("periodo_ancla") % 100)
    .withColumn("es_mes_ancla", 
                (col("Ano") == col("ano_ancla")) & (col("ORDENM") == col("mes_ancla")))
    .withColumn("antes_del_ancla", col("periodo_id") < col("periodo_ancla"))
    .withColumn("despues_del_ancla", col("periodo_id") > col("periodo_ancla"))
    .drop("periodo_candidato")
)

print("\n🎯 ANCLA GLOBAL DETECTADA:")
df_con_ancla.select("ano_ancla", "mes_ancla", "periodo_ancla").distinct().show()

print("\n[✓] SNIPPET 2 COMPLETADO")



StatementMeta(, 889760ac-d51b-408a-964a-45d2c97e687d, 4, Finished, Available, Finished)


SNIPPET 2: DETECCIÓN DEL MES ANCLA GLOBAL

🎯 ANCLA GLOBAL DETECTADA:
+---------+---------+-------------+
|ano_ancla|mes_ancla|periodo_ancla|
+---------+---------+-------------+
|     2026|        2|       202602|
+---------+---------+-------------+


[✓] SNIPPET 2 COMPLETADO


In [3]:
# ==============================================================================
# SNIPPET 3: MOTOR DE CÁLCULO CON FACTURACIÓN EN MES ANCLA
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 3: MOTOR DE CÁLCULO - PROPAGACIÓN MES A MES")
print("="*80)

w_continuo = Window.partitionBy("PAIS", "Familia", "Planificador", "PWOH").orderBy("Ano", "ORDENM")
w_año = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano")
w_año_pwoh = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano", "PWOH")

df_con_limites = (
    df_con_ancla
    .withColumn("tiene_sop", when(coalesce(col("Fcst"), lit(0.0)) > 0.0, col("ORDENM")))
    .withColumn("ultimo_mes_sop", f_max("tiene_sop").over(w_año))
    .withColumn("ultimo_mes_sop", coalesce(col("ultimo_mes_sop"), lit(12)))
    .drop("tiene_sop")
)

df_con_pwoh = (
    df_con_limites
    .withColumn("llegadas_original_daily", coalesce(col("Llegadas"), lit(0.0)))
    .withColumn("tiene_llegadas_daily", coalesce(col("Llegadas"), lit(0.0)) > 0.0)
    .withColumn("PWOH", explode(array([lit(i) for i in range(1, 11)])))
    .withColumn("sop_enero_mismo_año", 
                f_max(when(col("ORDENM") == 1, col("Fcst"))).over(w_año_pwoh))
)

print(f"✓ Registros después de expandir PWOH: {df_con_pwoh.count():,}")

df_preparado = df_con_pwoh

df_calculado = df_preparado

for iteracion in range(1, 16):
    print(f"  Iteración {iteracion}/15...")
    
    df_calculado = (
        df_calculado
        .withColumn("Inv_Final_anterior", lag("Inv_Final").over(w_continuo))
        .withColumn(
            "Inv_Inicial_nuevo",
            when(
                col("antes_del_ancla"),
                col("Inv_Inicial")
            ).otherwise(
                coalesce(col("Inv_Final_anterior"), col("Inv_Inicial"))
            )
        )
        .drop("Inv_Final_anterior")
    )
    
    df_calculado = (
        df_calculado
        .withColumn("S&OP_siguiente", lead("Fcst", 1).over(w_continuo))
        .withColumn(
            "Llegadas_nuevo",
            when(
                col("antes_del_ancla") & (coalesce(col("Fcst"), lit(0.0)) > 0.0),
                greatest(
                    lit(0.0),
                    coalesce(col("Inv_Final"), lit(0.0)) - 
                    coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) + 
                    coalesce(col("Fcst"), lit(0.0))
                )
            ).when(
                col("es_mes_ancla"),
                col("llegadas_original_daily")
            ).when(
                col("despues_del_ancla") & col("tiene_llegadas_daily"),
                col("llegadas_original_daily")
            ).when(
                col("despues_del_ancla") & 
                ~col("tiene_llegadas_daily") &
                (col("ORDENM") == 12) &
                (coalesce(col("Fcst"), lit(0.0)) > 0.0),
                greatest(
                    lit(0.0),
                    ((coalesce(col("sop_enero_mismo_año"), lit(0.0)) / 4.0) * col("PWOH"))
                    + coalesce(col("Fcst"), lit(0.0))
                    - coalesce(col("Inv_Inicial_nuevo"), lit(0.0))
                )
            ).when(
                col("despues_del_ancla") & 
                ~col("tiene_llegadas_daily") &
                (col("ORDENM") <= col("ultimo_mes_sop")) &
                (coalesce(col("Fcst"), lit(0.0)) > 0.0),
                greatest(
                    lit(0.0),
                    ((coalesce(col("S&OP_siguiente"), lit(0.0)) / 4.0) * col("PWOH"))
                    + coalesce(col("Fcst"), lit(0.0))
                    - coalesce(col("Inv_Inicial_nuevo"), lit(0.0))
                )
            ).otherwise(lit(0.0))
        )
        .drop("S&OP_siguiente")
    )
    
    df_calculado = df_calculado.withColumn(
        "Inv_Final_nuevo",
        when(
            col("antes_del_ancla"),
            coalesce(col("Inv_Final"), lit(0.0))
        ).when(
            col("es_mes_ancla"),
            greatest(
                lit(0.0),
                coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) +
                coalesce(col("Llegadas_nuevo"), lit(0.0)) -
                coalesce(col("Facturacion"), lit(0.0))
            )
        ).when(
            col("despues_del_ancla") &
            (col("ORDENM") <= col("ultimo_mes_sop")) &
            (coalesce(col("Fcst"), lit(0.0)) > 0.0),
            greatest(
                lit(0.0),
                coalesce(col("Inv_Inicial_nuevo"), lit(0.0)) +
                coalesce(col("Llegadas_nuevo"), lit(0.0)) -
                coalesce(col("Fcst"), lit(0.0))
            )
        ).otherwise(lit(0.0))
    )
    
    df_calculado = (
        df_calculado
        .drop("Inv_Inicial", "Llegadas", "Inv_Final")
        .withColumnRenamed("Inv_Inicial_nuevo", "Inv_Inicial")
        .withColumnRenamed("Llegadas_nuevo", "Llegadas")
        .withColumnRenamed("Inv_Final_nuevo", "Inv_Final")
    )

df_calculado = (
    df_calculado
    .filter(col("ORDENM") <= col("ultimo_mes_sop"))
    .drop("sop_enero_mismo_año", "llegadas_original_daily", "tiene_llegadas_daily")
)

print(f"✓ Registros después de filtrar: {df_calculado.count():,}")

df_calculado = df_calculado.cache()

print("\n[✓] SNIPPET 3 COMPLETADO")


StatementMeta(, 889760ac-d51b-408a-964a-45d2c97e687d, 5, Finished, Available, Finished)


SNIPPET 3: MOTOR DE CÁLCULO - PROPAGACIÓN MES A MES
✓ Registros después de expandir PWOH: 388,800
  Iteración 1/15...
  Iteración 2/15...
  Iteración 3/15...
  Iteración 4/15...
  Iteración 5/15...
  Iteración 6/15...
  Iteración 7/15...
  Iteración 8/15...
  Iteración 9/15...
  Iteración 10/15...
  Iteración 11/15...
  Iteración 12/15...
  Iteración 13/15...
  Iteración 14/15...
  Iteración 15/15...
✓ Registros después de filtrar: 380,620

[✓] SNIPPET 3 COMPLETADO


In [4]:
# ==============================================================================
# SNIPPET 4: CÁLCULO DE TTOS
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 4: CÁLCULO DE TTOS")
print("="*80)

w_intra_ano = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano", "PWOH").orderBy("ORDENM")
w_multi_ano = Window.partitionBy("PAIS", "Familia", "Planificador", "PWOH").orderBy("Ano", "ORDENM")

df_ttos = (
    df_calculado
    .withColumn("Lleg_t1_intra", lead("Llegadas", 1).over(w_intra_ano))
    .withColumn("Lleg_t2_intra", lead("Llegadas", 2).over(w_intra_ano))
    .withColumn("Lleg_t1_multi", lead("Llegadas", 1).over(w_multi_ano))
    .withColumn("Lleg_t2_multi", lead("Llegadas", 2).over(w_multi_ano))
    .withColumn(
        "TTOS",
        when(
            col("ORDENM") == 11,
            coalesce(col("Lleg_t1_intra"), lit(0.0)) +
            (coalesce(col("Lleg_t2_multi"), col("Lleg_t1_intra"), lit(0.0)) / 2.0)
        ).when(
            col("ORDENM") == 12,
            coalesce(col("Lleg_t1_multi"), lit(0.0)) +
            (coalesce(col("Lleg_t2_multi"), lit(0.0)) / 2.0)
        ).otherwise(
            coalesce(col("Lleg_t1_intra"), lit(0.0)) +
            (coalesce(col("Lleg_t2_intra"), lit(0.0)) / 2.0)
        )
    )
    .withColumn("UND_TTOS", F.round(col("Inv_Final") + col("TTOS"), 0))
    .drop("Lleg_t1_intra", "Lleg_t2_intra", "Lleg_t1_multi", "Lleg_t2_multi")
)

df_ttos = df_ttos.cache()

print(f"\n✓ TTOS calculado: {df_ttos.count():,} registros")

print("\n[✓] SNIPPET 4 COMPLETADO")


StatementMeta(, 889760ac-d51b-408a-964a-45d2c97e687d, 6, Finished, Available, Finished)


SNIPPET 4: CÁLCULO DE TTOS

✓ TTOS calculado: 380,620 registros

[✓] SNIPPET 4 COMPLETADO


In [5]:
# ==============================================================================
# SNIPPET 5: CÁLCULO DE WOH Y WOH_TTOS
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 5: CÁLCULO DE WOH Y WOH_TTOS")
print("="*80)

df_ultimo_fact = (
    df_ttos
    .filter(col("Facturacion") > 0)
    .groupBy("PAIS", "Familia", "Planificador", "Ano", "PWOH")
    .agg(f_max("ORDENM").alias("ultimo_fact"))
)

df_fallbacks = (
    df_ttos
    .filter((col("Ano") == 2025) & col("ORDENM").isin(1, 2))
    .groupBy("PAIS", "Familia", "Planificador", "PWOH")
    .pivot("ORDENM", [1, 2])
    .agg(F.first("Fcst"))
    .withColumnRenamed("1", "fcst_ene_2025")
    .withColumnRenamed("2", "fcst_feb_2025")
)

w_intra_woh = Window.partitionBy("PAIS", "Familia", "Planificador", "Ano", "PWOH").orderBy("ORDENM")

df_woh = (
    df_ttos
    .join(df_ultimo_fact, ["PAIS", "Familia", "Planificador", "Ano", "PWOH"], "left")
    .join(df_fallbacks, ["PAIS", "Familia", "Planificador", "PWOH"], "left")
    .withColumn("ultimo_fact", coalesce(col("ultimo_fact"), lit(0)))
    .fillna(0, ["fcst_ene_2025", "fcst_feb_2025"])
    .withColumn("fcst_m1", lead("Fcst", 1).over(w_intra_woh))
    .withColumn("fcst_m2", lead("Fcst", 2).over(w_intra_woh))
    .withColumn(
        "denominador",
        when(
            col("ORDENM") <= col("ultimo_fact"),
            when(
                col("ORDENM") == 11,
                (coalesce(col("fcst_m1"), lit(0.0)) + 
                 coalesce(col("fcst_ene_2025"), lit(0.0))) / 8.0
            ).when(
                col("ORDENM") == 12,
                (coalesce(col("fcst_ene_2025"), lit(0.0)) + 
                 coalesce(col("fcst_feb_2025"), lit(0.0))) / 8.0
            ).otherwise(
                (coalesce(col("fcst_m1"), lit(0.0)) + 
                 coalesce(col("fcst_m2"), lit(0.0))) / 8.0
            )
        ).otherwise(
            when(
                col("ORDENM") == 12,
                coalesce(col("fcst_ene_2025"), lit(0.0)) / 4.0
            ).otherwise(
                coalesce(col("fcst_m1"), lit(0.0)) / 4.0
            )
        )
    )
    .withColumn(
        "WOH",
        when((col("Inv_Final") <= 0) | (col("denominador") <= 0), lit(0))
        .otherwise(F.round(col("Inv_Final") / col("denominador"), 0).cast("int"))
    )
    .withColumn(
        "WOH_TTOS",
        when((col("UND_TTOS") <= 0) | (col("denominador") <= 0), lit(0))
        .otherwise(F.round(col("UND_TTOS") / col("denominador"), 0).cast("int"))
    )
    .withColumnRenamed("Inv_Inicial", "Inv_Inic")
    .drop("fcst_m1", "fcst_m2", "denominador", "fcst_ene_2025", "fcst_feb_2025")
)

df_woh = df_woh.cache()

print(f"\n✓ WOH calculado: {df_woh.count():,} registros")

print("\n[✓] SNIPPET 5 COMPLETADO")

StatementMeta(, 889760ac-d51b-408a-964a-45d2c97e687d, 7, Finished, Available, Finished)


SNIPPET 5: CÁLCULO DE WOH Y WOH_TTOS

✓ WOH calculado: 380,620 registros

[✓] SNIPPET 5 COMPLETADO


In [6]:
# ==============================================================================
# SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL
# ==============================================================================
print("\n" + "="*80)
print("SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL")
print("="*80)

try:
    df_bgt_pp = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_PP.csv")
        .select(col("Pais").alias("PAIS"), col("Familia"), 
                col("PP").cast("double").alias("BGT_PP"))
    )
    print(f"✓ BGT_PP cargado exitosamente: {df_bgt_pp.count():,} registros")
except:
    print("⚠ BGT_PP no encontrado, usando defaults")
    df_bgt_pp = spark.createDataFrame([
        ("CR", "LAVA", 164.5), ("CO", "LAVA", 100.0), ("MX", "SAGA", 120.0)
    ], ["PAIS", "Familia", "BGT_PP"])

try:
    df_bgt_cont = (
        spark.read.format("csv").option("header", "true").load("Files/BGT_CONT.csv")
        .select("PAIS", "Familia", "Attribute", col("ORDENM").cast("int"), 
                col("BGT_QTY").cast("double"), 
                col("BGT_$").cast("double").alias("BGT_USD"))
    )
    print(f"✓ BGT_CONT cargado exitosamente: {df_bgt_cont.count():,} registros")
except:
    print("⚠ BGT_CONT no encontrado, usando defaults")
    df_bgt_cont = spark.createDataFrame([
        ("CR", "LAVA", "ENE", 1, 2550.0, 100000.0),
        ("CR", "LAVA", "FEB", 2, 2219.0, 95000.0),
        ("CO", "LAVA", "ENE", 1, 1000.0, 80000.0),
        ("MX", "SAGA", "ENE", 1, 1500.0, 120000.0)
    ], ["PAIS", "Familia", "Attribute", "ORDENM", "BGT_QTY", "BGT_USD"])

df_con_monetarios = (
    df_woh
    .join(df_bgt_pp, ["PAIS", "Familia"], "left")
    .join(df_bgt_cont, ["PAIS", "Familia", "Attribute", "ORDENM"], "left")
    .fillna(0, ["BGT_PP", "BGT_QTY", "BGT_USD"])
    .withColumn("Inv_Final_USD", (col("Inv_Final") * col("BGT_PP")) / 1_000_000)
    .withColumn("TTOS_USD", (col("TTOS") * col("BGT_PP")) / 1_000_000)
    .withColumn("USD_OH_TTOS", col("Inv_Final_USD") + col("TTOS_USD"))
    .withColumn("Dif_Real_vs_BGT_QTY", col("UND_TTOS") - col("BGT_QTY"))
    .withColumn("Dif_Real_vs_BGT_USD", col("USD_OH_TTOS") - col("BGT_USD"))
)

print(f"✓ Monetarios agregados: {df_con_monetarios.count():,} registros")

df_tabla_final = (
    df_con_monetarios
    .select(
        col("Ano").cast("int"),
        col("Mes"),
        when(col("PAIS").isin("CO", "EC", "PE", "AR", "CL"), "Andino")
        .when(col("PAIS").isin("CR", "GT", "HN", "SV"), "CEAM")
        .when(col("PAIS").isin("MX"), "Mexico")
        .when(col("PAIS").isin("CA"), "NorteAmerica")
        .otherwise("Otros").alias("Region"),
        col("PAIS"),
        col("Familia"),
        col("Planificador"),
        col("Attribute"),
        col("Inv_Inic").cast("double"),
        col("Llegadas").cast("double"),
        col("Fcst").alias("S&PO").cast("double"),
        col("Facturacion").alias("Facturado").cast("double"),
        col("Inv_Final").cast("double"),
        F.round(col("Inv_Final_USD"), 4).alias("Inv_Final_$"),
        col("WOH").alias("WHO").cast("int"),
        col("TTOS").cast("double"),
        F.round(col("TTOS_USD"), 4).alias("TTOS_$"),
        F.round(col("USD_OH_TTOS"), 4).alias("USD_OH_TTOS"),
        col("UND_TTOS").alias("UND_TTO").cast("double"),
        col("WOH_TTOS").alias("WOH_TTO").cast("int"),
        col("BGT_QTY").cast("double"),
        col("BGT_USD").alias("BGT_$").cast("double"),
        col("Dif_Real_vs_BGT_QTY").cast("double"),
        col("Dif_Real_vs_BGT_USD").alias("Dif_Real_vs_BGT_$").cast("double"),
        col("PWOH").cast("double"),
        col("ORDENM").cast("double"),
        lit("Facturado").alias("Calculo")
    )
)

print(f"✓ Tabla final estructurada: {df_tabla_final.count():,} registros")

count_antes = df_tabla_final.count()

df_tabla_sin_duplicados = df_tabla_final.dropDuplicates([
    "PAIS", "Familia", "Planificador", "Ano", "ORDENM", "PWOH"
])

count_despues = df_tabla_sin_duplicados.count()
duplicados_eliminados = count_antes - count_despues

print(f"✓ Registros antes: {count_antes:,}")
print(f"✓ Registros después: {count_despues:,}")
print(f"✓ Duplicados eliminados: {duplicados_eliminados:,}")

df_tabla_final_opt = df_tabla_sin_duplicados.repartition(200, "PAIS", "Familia", "Planificador", "PWOH")

spark.sql("USE LH_CARATULAS_PLNF")

try:
    spark.sql("DROP TABLE IF EXISTS tbl_caratula_cdv_planificador_facturado")
    print("✓ Tabla anterior eliminada")
except Exception as e:
    print(f"⚠ No se pudo eliminar tabla anterior: {e}")

df_tabla_final_opt.write.format("delta").mode("overwrite").saveAsTable("tbl_caratula_cdv_planificador_facturado")

print("\n✅ TABLA FINAL GUARDADA:")
print("  • Nombre: tbl_caratula_cdv_planificador_facturado")
print("  • Esquema: LH_CARATULAS_PLNF")
print(f"  • Registros finales: {count_despues:,}")

print("\n[✓] SNIPPET 6 COMPLETADO")


StatementMeta(, 889760ac-d51b-408a-964a-45d2c97e687d, 8, Finished, Available, Finished)


SNIPPET 6: MONETARIOS, BUDGET Y GUARDADO FINAL
✓ BGT_PP cargado exitosamente: 32 registros
✓ BGT_CONT cargado exitosamente: 2,052 registros
✓ Monetarios agregados: 622,340 registros
✓ Tabla final estructurada: 622,340 registros
✓ Registros antes: 622,340
✓ Registros después: 380,620
✓ Duplicados eliminados: 241,720
✓ Tabla anterior eliminada

✅ TABLA FINAL GUARDADA:
  • Nombre: tbl_caratula_cdv_planificador_facturado
  • Esquema: LH_CARATULAS_PLNF
  • Registros finales: 380,620

[✓] SNIPPET 6 COMPLETADO
